# CNN Tile Classifier — Final
Trains `customCNN` for GridAdventure tile classification:
- Single synthetic level containing all 13 entity types
- 500 seeds, tile crops labelled via `appearance.name`
- Each tile saved twice: original + ColorJitter augmentation
- Output: `get_model()` snippet to paste into agent

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/TIMOTHY NUS/Y2S2/CS2109S/capstone-project'

import sys
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU — go to Runtime, Change runtime type, GPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

In [ ]:
!pip install -q git+https://github.com/grid-universe/grid-adventure-v1.git
print('Dependencies ready')

## Generate labelled tile crops
Builds one synthetic level that contains every entity type, renders it under 500 different seeds, slices each render into per-cell tiles, and labels each tile using `entity.appearance.name`.

In [ ]:
from grid_adventure.grid import GridState, from_state
from grid_adventure.entities import (
    FloorEntity, WallEntity, ExitEntity, CoinEntity, GemEntity, KeyEntity,
    LockedDoorEntity, LavaEntity, BoxEntity, SpeedPowerUpEntity,
    ShieldPowerUpEntity, PhasingPowerUpEntity, create_agent_entity,
)
from grid_adventure.movements import MOVEMENTS
from grid_adventure.objectives import OBJECTIVES
from utils import create_env
import numpy as np
from collections import Counter

LABEL_MAP = {'floor': 0, 'wall': 1, 'human': 2, 'exit': 3, 'coin': 4, 'gem': 5, 'key': 6, 'door': 7, 'lava': 8, 'box': 9, 'boots': 10, 'shield': 11, 'ghost': 12}
NUM_CLASSES = len(LABEL_MAP)
print(f'{NUM_CLASSES} classes:', list(LABEL_MAP.keys()))


def build_level() -> GridState:
    gridstate = GridState(
        width=4, height=7,
        movement=MOVEMENTS['cardinal'],
        objective=OBJECTIVES['collect_gems_and_exit'],
        seed=0,
    )
    for (x, y) in [(0, 2), (1, 5)]:
        gridstate.add((x, y), BoxEntity())
    for (x, y) in [(0, 1), (1, 4)]:
        gridstate.add((x, y), CoinEntity())
    for (x, y) in [(0, 4), (3, 0)]:
        gridstate.add((x, y), ExitEntity())
    for (x, y) in [
        (0,0),(0,1),(0,2),(0,3),(0,4),(0,5),(0,6),
        (1,0),(1,1),(1,2),(1,3),(1,4),(1,5),(1,6),
        (2,0),(2,1),(2,2),(2,3),(2,4),(2,5),(2,6),
        (3,0),(3,1),(3,2),(3,3),(3,4),(3,5),(3,6),
    ]:
        gridstate.add((x, y), FloorEntity())
    for (x, y) in [(1, 1), (2, 4)]:
        gridstate.add((x, y), GemEntity())
    for (x, y) in [(2, 1), (3, 4)]:
        gridstate.add((x, y), KeyEntity())
    for (x, y) in [(1, 2), (2, 5)]:
        gridstate.add((x, y), LavaEntity())
    for (x, y) in [(0, 5), (3, 1)]:
        gridstate.add((x, y), LockedDoorEntity())
    for (x, y) in [(0, 3), (1, 6)]:
        gridstate.add((x, y), PhasingPowerUpEntity())
    for (x, y) in [(0, 6), (3, 2)]:
        gridstate.add((x, y), ShieldPowerUpEntity())
    for (x, y) in [(2, 2), (3, 5)]:
        gridstate.add((x, y), SpeedPowerUpEntity())
    for (x, y) in [(1, 0), (2, 3)]:
        gridstate.add((x, y), WallEntity())
    for (x, y) in [(2, 0), (3, 3), (3, 6)]:
        gridstate.add((x, y), create_agent_entity(health=5))
    return gridstate


def get_entity_label(gs, x, y):
    for obj in gs.objects_at((x, y)):
        n = type(obj).__name__.lower()
        if 'agent' in n: return 'human'
        if 'wall' in n: return 'wall'
        if 'exit' in n: return 'exit'
        if 'lava' in n: return 'lava'
        if 'gem' in n: return 'gem'
        if 'locked' in n: return 'door'
        if 'key' in n: return 'key'
        if 'coin' in n: return 'coin'
        if 'box' in n: return 'box'
        if 'speed' in n or 'boot'  in n: return 'boots'
        if 'phas' in n or 'ghost' in n: return 'ghost'
        if 'shield' in n: return 'shield'
    return 'floor'


def create_labels(num_seeds=500):
    tiles = []
    for seed in range(num_seeds):
        env = create_env(build_level, observation_type='image', seed=seed)
        image_obs, _ = env.reset()
        assert isinstance(image_obs, dict), "expected image observation"
        image = image_obs['image']  # (H, W, 4) RGBA uint8

        img_h, img_w = image.shape[0], image.shape[1]
        assert env.state is not None
        gs = from_state(env.state)
        grid_w, grid_h = gs.width, gs.height
        tile_w = img_w // grid_w
        tile_h = img_h // grid_h

        for y in range(grid_h):
            for x in range(grid_w):
                tile_img = image[
                    y * tile_h : (y + 1) * tile_h,
                    x * tile_w : (x + 1) * tile_w,
                ]
                label = get_entity_label(gs, x, y)
                tiles.append((tile_img, label))
        if (seed + 1) % 100 == 0:
            print(f'  {seed + 1}/{num_seeds} seeds processed')
    return tiles


print('Generating tiles from 500 seeds')
tiles = create_labels(num_seeds=500)
print(f'Total raw tiles: {len(tiles)}')

label_counts = Counter(label for _, label in tiles)
print('Label distribution:')
for lbl, cnt in sorted(label_counts.items()):
    print(f'{lbl:10s}: {cnt}')

## Preprocess and save dataset
Each tile is stored twice: once resized as-is, once with ColorJitter(augmented). 
Both saved to a single `.pt` file for training.

In [ ]:
from torchvision import transforms
from PIL import Image as PILImage

IMG_SIZE = 64
PT_PATH = 'processed_tiles_final.pt'

augment = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
    transforms.ToTensor(),  # [0, 1] float, shape (4, H, W)
])

normal = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

processed = []
for img, label in tiles:
    processed.append((normal(img),  LABEL_MAP[label]))
    processed.append((augment(img), LABEL_MAP[label]))

images = torch.stack([p[0] for p in processed])
labels = torch.tensor([p[1] for p in processed], dtype=torch.long)

torch.save({'images': images, 'labels': labels}, PT_PATH)
print(f'Saved {len(labels)} tiles to {PT_PATH}')
print(f'Tensor shape: {images.shape}  ({images.element_size() * images.nelement() / 1e6:.1f} MB)')

## CNN Model Architecture
`customCNN`: 3 conv blocks, BatchNorm + LeakyReLU(0.1), MaxPool after first 2 blocks, Global Average Pooling, then a two-layer FC head with Dropout(0.3)

In [ ]:
import torch.nn as nn

class customCNN(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
        )
        self.fcn = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.shape[0], 128, -1).mean(2)  # Global Average Pooling
        x = self.fcn(x)
        return x

# Sanity check
model = customCNN(num_classes=NUM_CLASSES).to(device)
dummy = torch.randn(2, 4, IMG_SIZE, IMG_SIZE).to(device)
out = model(dummy)
assert out.shape == (2, NUM_CLASSES), f'Unexpected output shape: {out.shape}'

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}  ({total_params * 4 / 1e6:.2f} MB)')

## CNN Model Training
Adam lr=0.001, StepLR (×0.1 every 15 epochs), CrossEntropyLoss, 50 epochs, batch size 64

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PreprocessedTileDataset(Dataset):
    def __init__(self, path):
        data = torch.load(path)
        self.images = data['images']
        self.labels = data['labels']
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


dataset = PreprocessedTileDataset(PT_PATH)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f'Dataset: {len(dataset)} samples, {len(loader)} batches/epoch')

model = customCNN(num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
criterion = nn.CrossEntropyLoss()

model.train()
EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS}  loss={epoch_loss:.4f}  lr={optimizer.param_groups[0]["lr"]:.6f}')

print('Training complete.')

## Accuracy Check

In [ ]:
model.eval()
correct = total = 0
class_correct = [0] * NUM_CLASSES
class_total = [0] * NUM_CLASSES
idx_to_label = {v: k for k, v in LABEL_MAP.items()}

with torch.no_grad():
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        preds = model(x_batch).argmax(1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
        for pred, label in zip(preds.cpu().tolist(), y_batch.cpu().tolist()):
            class_total[label] += 1
            class_correct[label] += int(pred == label)

print(f'Overall accuracy: {correct/total:.4f} ({correct}/{total})')
print('Per-class accuracy:')
for idx in range(NUM_CLASSES):
    n = class_total[idx]
    if n == 0: continue
    acc = class_correct[idx] / n
    marker = 'CORRECT:' if acc >= 0.90 else 'WRONG:'
    print(f' {marker} {idx_to_label[idx]:10s}: {acc:.3f} ({class_correct[idx]}/{n})')

## Export `get_model()` snippet

In [ ]:
from utils import generate_torch_loader_snippet

example_input = torch.randn(1, 4, IMG_SIZE, IMG_SIZE).to(device)
snippet = generate_torch_loader_snippet(model, example_inputs=example_input)

print(snippet)
print(f'\nSnippet size: {len(snippet) / 1024:.1f} KB')